# Week 12 - Sentiment Analysis (2/2)
**Notebook 2 : The Machine Learning-based Approach**

**Unstructured Data Analysis (2026-2)** · Professor: Misuk Kim · Teaching Assistant: Hojin Son

Notebook 1 ended with a problem: counting opinion words gives every word the same single vote. Here the votes are
**learned**. The notebook follows chapter 4 of lecture 11-1 / 11-2 (below "lecture 11") and the supervised formulation of
lecture 10-2, on the same 7,500 hotel reviews and the same 1,352 test reviews.

| Lecture | What we do in code |
|---|---|
| 10-2 p. 24-25 Document sentiment classification | Naive Bayes and logistic regression on word counts, unigrams vs bigrams |
| 11 p. 22-24 Regression-based approach | Elastic Net predicts the rating, and its coefficients become a sentiment lexicon |
| 11 p. 29-35, 44 Semi-supervised approach | label spreading on the lecture's word graph, then on a Word2Vec graph of 6,000 words |
| (all) | every method of the two notebooks on the same test reviews |

> **How to use this notebook in Colab**: `File ▸ Save a copy in Drive`, run the cells from top to bottom with `Shift + Enter`.
> The **🤔 Quick check** notes say what to look for in an output. The two cells marked **🔧** ask you to change one value
> and run the cell again.

## &nbsp;0. Setup
Gensim (Week 6) trains the Word2Vec model of §3. Everything else is pre-installed in Colab.

> ⚠️ If Colab asks you to **restart the session** after the install, do it (`Runtime ▸ Restart session`) and run the
> cells again from here.

In [ ]:
!pip install -q gensim

import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scipy.sparse as sp
from gensim.models import Word2Vec
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression, ElasticNet, ElasticNetCV
from sklearn.neighbors import kneighbors_graph
from sklearn.metrics import accuracy_score, f1_score, mean_absolute_error

import nltk
nltk.download('opinion_lexicon', quiet=True)
from nltk.corpus import opinion_lexicon

%config InlineBackend.figure_format = 'retina'
%matplotlib inline
pd.set_option('display.width', 140)

In [ ]:
DATA_URL = "https://raw.githubusercontent.com/snhzyn/2026-2-Unstructured-Data-Analysis/main/week12-sentiment-analysis/data/tripadvisor_sample.csv"

reviews = pd.read_csv(DATA_URL)
STAR_TO_SENTIMENT = {1: 'negative', 2: 'negative', 3: 'neutral', 4: 'positive', 5: 'positive'}
reviews['sentiment'] = reviews['rating'].map(STAR_TO_SENTIMENT)
train = reviews[reviews['split'] == 'train'].reset_index(drop=True)
test = reviews[reviews['split'] == 'test'].reset_index(drop=True)

train_pn = train[train['sentiment'] != 'neutral'].reset_index(drop=True)     # neutral is ignored (10-2 p. 25)
test_pn = test[test['sentiment'] != 'neutral'].reset_index(drop=True)
y_train, y_true = train_pn['sentiment'], test_pn['sentiment']
print('train :', len(train), '(positive/negative', len(train_pn), ')   test :', len(test), '(positive/negative', len(test_pn), ')')

The helpers of Notebook 1: a tokenizer, the counting rule of lecture 11 p. 44, and the opinion lexicon as the baseline to beat.

In [ ]:
def tokenize(text):
    return re.findall(r"[a-z][a-z'-]*", text.lower())


def lexicon_classify(text, pos_words, neg_words):
    words = set(tokenize(text))
    return 'positive' if len(words & pos_words) > len(words & neg_words) else 'negative'


def report(name, y_true, y_pred):
    f1_neg, f1_pos = f1_score(y_true, y_pred, labels=['negative', 'positive'], average=None)
    acc = accuracy_score(y_true, y_pred)
    print(f'{name:30} accuracy {acc:.3f}   F1 negative {f1_neg:.3f}   F1 positive {f1_pos:.3f}')
    results[name] = (acc, f1_neg)


def confusion(y_true, y_pred):
    return pd.crosstab(pd.Series(y_true, name='actual'), pd.Series(y_pred, name='prediction'))


results = {}                                    # every method lands here, compared in section 4
POS_WORDS, NEG_WORDS = set(opinion_lexicon.positive()), set(opinion_lexicon.negative())
report('Opinion lexicon (count)', y_true, [lexicon_classify(t, POS_WORDS, NEG_WORDS) for t in test_pn['text']])

## &nbsp;1. Document sentiment classification as supervised learning
*Supervised Learning Formulation* (lecture 10-2 p. 25): training and test data with star ratings, and *it is basically a text
classification problem* (p. 24). So the classifiers of Week 10 apply directly. **Key: feature engineering**, and the
lecture's first feature is *typically unigrams (bag of individual words)*.

### &nbsp;1-1. Naive Bayes and logistic regression on word counts

In [ ]:
vec = CountVectorizer(min_df=2)                           # unigrams in at least 2 training reviews
X_train = vec.fit_transform(train_pn['text'])
X_test = vec.transform(test_pn['text'])
print('X_train :', X_train.shape, '  X_test :', X_test.shape)

nb = MultinomialNB().fit(X_train, y_train)
lr = LogisticRegression(max_iter=2000).fit(X_train, y_train)
pred_nb, pred_lr = nb.predict(X_test), lr.predict(X_test)

report('Naive Bayes (unigram)', y_true, pred_nb)
report('Logistic regression (unigram)', y_true, pred_lr)
confusion(y_true, pred_lr)

**🤔 Quick check** Both classifiers beat the lexicon, and the gain is in the **negative** class: F1 0.736 → about 0.87.
The price is the labels. These models needed 5,389 reviews whose sentiment we already knew.

Logistic regression gives every word one weight. A positive weight pushes a review toward *positive*, a negative
weight toward *negative*. The words with the largest weights on each side:

In [ ]:
names = vec.get_feature_names_out()
order = np.argsort(lr.coef_[0])                           # coef > 0 pushes toward 'positive'
pd.DataFrame({'most negative': names[order[:12]], 'weight ': lr.coef_[0][order[:12]].round(2),
              'most positive': names[order[::-1][:12]], 'weight': lr.coef_[0][order[::-1][:12]].round(2)})

**🤔 Quick check** *ok* (−1.16) is as negative as *awful*: the lukewarm word of Notebook 1 again. *bit* is positive
(*"a bit pricey"*, *"a bit small"* are written by happy guests). *not* and *no* get −0.70 each, so one *not* no longer
outvotes *perfect* (+1.55), as it did in the counting rule.

### &nbsp;1-2. 🔧 Feature engineering : unigrams vs bigrams
The lecture's list of features also has *negations* (p. 25). A bigram feature such as *not clean* keeps the negation with
its word. Run the cell, then **change `NGRAM = (1, 1)` to `NGRAM = (1, 2)`** and run it again.

In [ ]:
NGRAM = (1, 1)                                  # 🔧 change to (1, 2) and run again

vec_ng = CountVectorizer(min_df=2, ngram_range=NGRAM)
Xn_train, Xn_test = vec_ng.fit_transform(train_pn['text']), vec_ng.transform(test_pn['text'])
lr_ng = LogisticRegression(max_iter=2000).fit(Xn_train, y_train)

print('features :', Xn_train.shape[1])
name = 'Logistic regression (unigram)' if NGRAM == (1, 1) else f'Logistic regression {NGRAM}'
report(name, y_true, lr_ng.predict(Xn_test))
for f in ['clean', 'not clean', 'recommend', 'not recommend', 'worth', 'not worth', 'problem', 'no problem']:
    if f in vec_ng.vocabulary_:
        print(f'   {f:15} {lr_ng.coef_[0][vec_ng.vocabulary_[f]]:6.2f}')

**🤔 Quick check** With bigrams the features grow from 11,159 to 54,990, accuracy goes from 0.956 to 0.962 and F1 of the
negative class from 0.862 to 0.883. The weights show why: *clean* +0.73 and *not clean* −0.55, *recommend* +0.37 and
*not recommend* −0.83, *problem* −0.12 and *no problem* +0.23. The negation is now part of the feature.

## &nbsp;2. Regression-based approach : a lexicon learned from the ratings
*Regression-based Approach* (lecture 11 p. 22-24): a document-term matrix of reviews and their **5-point ratings**.

* Response variable : the rating of a review
* Explanatory variables : the words used in the review
* **The regression coefficient of a word is its sentiment score**

The lecture fits an **Elastic Net** with 10-fold cross-validation:

$$\hat\beta = \arg\min_\beta \; \|y - X\beta\|^2 + \lambda_2 \|\beta\|^2 + \lambda_1 \|\beta\|_1$$

The $\ell_1$ part sets most coefficients to exactly zero (a *sparse* model), and the squared part keeps groups of similar words
together. Unlike §1, the 3-star reviews are used too: the response is the rating itself, not a class.

In [ ]:
vec_bin = CountVectorizer(binary=True, min_df=5)          # 1 if the review contains the word
D_train = vec_bin.fit_transform(train['text'])            # all 6,000 training reviews, 3 stars included
D_test = vec_bin.transform(test['text'])
words = vec_bin.get_feature_names_out()

cv = ElasticNetCV(l1_ratio=0.5, alphas=np.logspace(-3.5, -1.5, 15), cv=10, max_iter=5000)
cv.fit(D_train, train['rating'])                          # 10-fold cross-validation picks the penalty
en = ElasticNet(alpha=cv.alpha_, l1_ratio=0.5, max_iter=5000).fit(D_train, train['rating'])

coef = pd.Series(en.coef_, index=words)
print(f'document-term matrix : {D_train.shape}   best alpha : {cv.alpha_:.5f}')
print(f'words with a non-zero coefficient : {(coef != 0).sum()}  ({(coef > 0).sum()} positive, {(coef < 0).sum()} negative)')
pd.DataFrame({'most negative': coef.nsmallest(10).index, 'score ': coef.nsmallest(10).values.round(3),
              'most positive': coef.nlargest(10).index, 'score': coef.nlargest(10).values.round(3)})

**🤔 Quick check** Out of 6,380 words only about 800 keep a score, and the rest are exactly 0: this is the lexicon.
Each score is estimated together with the other words of the reviews, so it is the effect of the word given the rest. One
*terrible* costs about 0.6 stars. Look up the words that troubled Notebook 1:

In [ ]:
coef[['good', 'fine', 'ok', 'decent', 'pricey', 'not', 'no', 'dirty', 'small']].round(3).to_frame('Elastic Net score').T

**🤔 Quick check** *good* and *fine* get exactly 0, *ok* and *decent* are negative, and *pricey* is positive, the same
domain-dependent picture as the t-test of Notebook 1, but now with a size for each word.

*Sentiment lexicon performance* (p. 23) reports the error of the predicted ratings (MAE) and their correlation (COR)
with the true ratings. A predicted rating of 3 or more is read as *positive*.

In [ ]:
rating_pred = en.predict(D_test)
print(f'MAE {mean_absolute_error(test["rating"], rating_pred):.3f}   '
      f'COR {np.corrcoef(test["rating"], rating_pred)[0, 1]:.3f}   (all 1,500 test reviews)')

is_pn = (test['sentiment'] != 'neutral').to_numpy()
pred_en = np.where(rating_pred[is_pn] >= 3, 'positive', 'negative')
report('Elastic Net lexicon', y_true, pred_en)
confusion(y_true, pred_en)

### &nbsp;2-1. 🔧 How sparse is the lexicon?
`l1_ratio` sets the mix of the two penalties: 1.0 is the Lasso (only $\ell_1$), values near 0 are close to ridge
regression. The penalty `alpha` stays the same, so this shows how sparse the lexicon gets, not the best model for
each ratio. Run the cell, then **change `L1_RATIO = 0.5` to `1.0`, then to `0.1`**, and run it each time.

In [ ]:
L1_RATIO = 0.5                                  # 🔧 change to 1.0, then to 0.1

en_r = ElasticNet(alpha=cv.alpha_, l1_ratio=L1_RATIO, max_iter=5000).fit(D_train, train['rating'])
print(f'l1_ratio {L1_RATIO} : {(en_r.coef_ != 0).sum()} words in the lexicon   '
      f'MAE {mean_absolute_error(test["rating"], en_r.predict(D_test)):.3f}')

**🤔 Quick check** 839 words at 0.5, 432 with the Lasso and 3,364 at 0.1, while the MAE stays between 0.615 and 0.633. The
$\ell_1$ part *generates a sparse model*, and the squared part *removes the limitation on the number of selected variables* (p. 24).

## &nbsp;3. Semi-supervised approach : spread a few seed labels over a word graph
§1 and §2 needed a label or a rating for every training review. The semi-supervised approach of lecture 11 (p. 29-35)
needs only **a handful of labeled words**: embed the words (Word2Vec), connect each word to its nearest words (a graph),
and let the known labels spread along the edges (*graph-based SSL*).

### &nbsp;3-1. The lecture's example graph
*An illustrative example of label propagation/spreading* (p. 33-34). Four positive and four negative seed words.
Every step, each word takes the **average score of its neighbors**, and the seeds keep pulling toward their own label:

**new score = α × (average score of the neighbors) + (1 − α) × seed label**

The seed label is +1, −1 or 0, and $\alpha = 0.8$. In the code, `W` is the adjacency matrix of the graph and `deg`
the number of neighbors of each word.

In [ ]:
EDGES = [('song', 'burst'), ('song', 'fun'), ('burst', 'good'), ('good', 'awesome'), ('good', 'fun'),
         ('good', 'entertaining'), ('fun', 'entertaining'), ('awesome', 'sad'), ('awesome', 'excellent'),
         ('excellent', 'entertaining'), ('excellent', 'heartfelt'), ('sad', 'director'), ('sad', 'heartfelt'),
         ('heartfelt', 'handsome'), ('entertaining', 'handsome'), ('handsome', 'protagonist'),
         ('entertaining', 'watch'), ('watch', 'forced'), ('forced', 'mediocre'), ('mediocre', 'annoying'),
         ('superficial', 'annoying'), ('forced', 'annoying'), ('annoying', 'lacking'), ('lacking', 'disappointing'),
         ('annoying', 'boring'), ('forced', 'dirty'), ('dirty', 'boring'), ('dirty', 'friend'),
         ('boring', 'disappointing'), ('disappointing', 'insufficient'), ('boring', 'weak'),
         ('insufficient', 'weak'), ('weak', 'camera')]
SEEDS = {'good': 1, 'awesome': 1, 'entertaining': 1, 'excellent': 1,
         'annoying': -1, 'forced': -1, 'boring': -1, 'disappointing': -1}

toy_words = sorted({w for e in EDGES for w in e})
toy_index = {w: i for i, w in enumerate(toy_words)}
W_toy = np.zeros((len(toy_words), len(toy_words)))
for a, b in EDGES:
    W_toy[toy_index[a], toy_index[b]] = W_toy[toy_index[b], toy_index[a]] = 1
y_toy = np.array([SEEDS.get(w, 0) for w in toy_words], dtype=float)


def spread(W, y, alpha=0.8, n_steps=30):
    W = sp.csr_matrix(W)
    deg = np.asarray(W.sum(axis=1)).ravel()
    P = sp.diags(1 / deg) @ W                             # divide by the degree : average over the neighbors
    F = y.copy()
    for _ in range(n_steps):
        F = alpha * (P @ F) + (1 - alpha) * y
    return F


toy = pd.DataFrame({'seed': y_toy, 'after 1 step': spread(W_toy, y_toy, n_steps=1),
                    'after 30 steps': spread(W_toy, y_toy, n_steps=30)}, index=toy_words)
toy[toy['seed'] == 0].sort_values('after 1 step').round(3).T

**🤔 Quick check** After **one** step you get the picture of p. 34. The neighbors of the seeds take their color: *sad*,
*heartfelt*, *handsome*, *burst*, *fun* turn positive, and *mediocre*, *dirty*, *lacking*, *weak* turn negative. *watch* stays at 0
because it touches one positive and one negative seed, and the far words (*song*, *director*, *camera*, ...) are still 0.
After 30 steps the labels have reached every word. *watch* stays close to 0.

In [ ]:
POSITIONS = {'song': (4, 9), 'burst': (2.9, 8), 'fun': (5.1, 8.1), 'good': (4, 7.5), 'awesome': (2.9, 6.6),
             'entertaining': (5.4, 6.6), 'sad': (2.5, 5.7), 'excellent': (4, 5.4), 'heartfelt': (4, 4.3),
             'handsome': (5.5, 4.9), 'director': (2.5, 3.6), 'protagonist': (5.5, 3.6), 'watch': (7.2, 6.6),
             'superficial': (10.2, 8.6), 'mediocre': (9, 8), 'lacking': (11.4, 8.3), 'annoying': (10.2, 7.4),
             'forced': (9, 6.6), 'disappointing': (11.5, 6.6), 'boring': (10.2, 5.7), 'dirty': (9, 5.3),
             'insufficient': (12, 5.4), 'weak': (11.1, 4.6), 'friend': (9.7, 4.2), 'camera': (11.8, 3.7)}

fig, axes = plt.subplots(1, 2, figsize=(15, 4.6))
for ax, col in zip(axes, ['after 1 step', 'after 30 steps']):
    for a, b in EDGES:
        ax.plot(*zip(POSITIONS[a], POSITIONS[b]), color='lightgray', zorder=1)
    for w, (x, yy) in POSITIONS.items():
        s = toy.loc[w, col]
        color = 'tab:blue' if s > 0.01 else 'tab:red' if s < -0.01 else 'black'
        ax.text(x, yy, w, color=color, ha='center', va='center', fontsize=11,
                fontweight='bold' if w in SEEDS else 'normal', alpha=1 if w in SEEDS or abs(s) <= 0.01 else 0.75,
                bbox=dict(facecolor='white', edgecolor='none', pad=1), zorder=2)
    ax.set_title(f'{col}  (blue > 0, red < 0, black = about 0, bold = seed)')
    ax.set_xlim(1.5, 13); ax.set_ylim(3, 9.6); ax.axis('off')
plt.tight_layout()
plt.show()

### &nbsp;3-2. Word vectors from the reviews (Word2Vec, Week 6)
The lecture's framework (p. 29): *data preparation → Word2Vec → construct network → graph-based SSL → performance evaluation*.
We train a Skip-gram model on the 6,000 training reviews (the ratings are not used). This takes about 20 seconds.
With `seed=42` and one worker the numbers usually match the slides; another gensim version or environment can
give slightly different numbers.

In [ ]:
sentences = [tokenize(t) for t in train['text']]
w2v = Word2Vec(sentences, vector_size=100, window=5, min_count=5, sg=1, epochs=10, seed=42, workers=1)
vocab = w2v.wv.index_to_key
print('words with a vector :', len(vocab))
for w in ['dirty', 'spacious']:
    print(f'{w:9} ->', [s for s, _ in w2v.wv.most_similar(w, topn=8)])

### &nbsp;3-3. The word graph and the seed words
*Construct network*: every word is connected to its 10 nearest words (cosine similarity). The seeds are the
**pre-labeled sentiment words** of p. 32 (labeled by hand). Not all of them are used by hotel guests, so only the ones in our vocabulary count.

In [ ]:
PRE_POS = ['beautiful', 'wonderful', 'fabulous', 'amazing', 'delightful', 'splendid', 'happy', 'stunning',
           'breathtaking', 'gorgeous', 'incredible', 'inspiring', 'fantastic', 'joyful', 'exceptional', 'ecstatic',
           'charming', 'enchanting', 'pleasant', 'radiant', 'exquisite', 'marvelous', 'excellent', 'thrilling']
PRE_NEG = ['cheating', 'deceitful', 'disgusting', 'annoying', 'frustrating', 'awful', 'bad', 'terrible',
           'distressing', 'accursed', 'cursed', 'offensive', 'overblown', 'exaggerated', 'dreadful', 'perplexing',
           'confusing', 'unpleasant', 'impolitely', 'rude', 'abominable', 'hideous', 'ugly', 'horrendous']

index = {w: i for i, w in enumerate(vocab)}
y_seed = np.zeros(len(vocab))
for w in PRE_POS:
    if w in index: y_seed[index[w]] = 1
for w in PRE_NEG:
    if w in index: y_seed[index[w]] = -1
print('seeds in the vocabulary :', int((y_seed > 0).sum()), 'positive,', int((y_seed < 0).sum()), 'negative')

G = kneighbors_graph(w2v.wv.get_normed_vectors(), n_neighbors=10, metric='cosine')
W_words = ((G + G.T) > 0).astype(float)                   # undirected: an edge if either word is a neighbor
print('graph :', W_words.shape[0], 'words,', int(W_words.sum() // 2), 'edges')

for steps in [1, 2, 3, 5, 30]:
    print(f'after {steps:2} steps : {np.count_nonzero(spread(W_words, y_seed, n_steps=steps)):5} words have a score')

**🤔 Quick check** 30 seed words, and after 5 steps every one of the 6,485 words has a score, as in the large network of p. 35.

In [ ]:
F = spread(W_words, y_seed, n_steps=30)
score = pd.Series(F / np.abs(F).max(), index=vocab)       # scaled to [-1, 1] as on p. 40
learned = score[y_seed == 0]                              # leave the seeds out

pd.DataFrame({'most negative': learned.nsmallest(15).index, 'score ': learned.nsmallest(15).values.round(3),
              'most positive': learned.nlargest(15).index, 'score': learned.nlargest(15).values.round(3)})

**🤔 Quick check** None of these words was given. The positive side is general praise (*magnificent*, *superb*, even the
misspelled *excellant*), and the negative side found hotel complaints on its own (*stale*, *dirty*, *moldy*, *poisoning*,
*complained*). A few words are noise (*landscape*, *cherry*, *appears*): the graph only knows which words are used in
similar places. The learned scores stay well below the seeds (the largest is about 0.3), because a learned word is further
from the seeds than the seeds themselves.

### &nbsp;3-4. Performance evaluation
*Performance evaluation (word sentiment)* (p. 41): compare the sign of the learned score with the opinion lexicon, on the
words that are in both. Then *review polarity*, with the procedure of p. 44: count the positive and negative words of a review.

In [ ]:
in_lexicon = [w for w in learned.index if w in POS_WORDS or w in NEG_WORDS]
truth = np.array([w in POS_WORDS for w in in_lexicon])
print(f'word sentiment : {len(in_lexicon)} words in the opinion lexicon, '
      f'agreement {accuracy_score(truth, learned[in_lexicon] > 0):.3f}')

ssl_pos, ssl_neg = set(score.index[score > 0]), set(score.index[score < 0])
pred_ssl = [lexicon_classify(t, ssl_pos, ssl_neg) for t in test_pn['text']]
report('Label spreading lexicon', y_true, pred_ssl)
confusion(y_true, pred_ssl)

**🤔 Quick check** About three out of four lexicon words get the same sign as in the opinion lexicon (an agreement, not an accuracy on
true labels), and the reviews are classified with no review
label at all, from 30 words. The lecture also shows **self-training** for the same task (p. 36-38): train a classifier on
the seed words, label the words it is most confident about, and retrain. We do not code it today.

## &nbsp;4. Every method on the same 1,352 test reviews
The corpus dictionary row comes from Notebook 1 and is typed in ($\alpha = 0.05$, even if you changed `ALPHA` there).

In [ ]:
results['Corpus dictionary (count)'] = (0.798, 0.615)          # Notebook 1, section 3-4
NEEDS = {'Opinion lexicon (count)': 'a public word list', 'Corpus dictionary (count)': 'ratings',
         'Label spreading lexicon': '30 seed words', 'Elastic Net lexicon': 'ratings'}

comparison = pd.DataFrame(results, index=['accuracy', 'F1 negative']).T
comparison['needs'] = [NEEDS.get(m, 'labeled reviews') for m in comparison.index]
comparison = comparison.sort_values('accuracy').round(3)

fig, ax = plt.subplots(figsize=(10, 4.2))
y = np.arange(len(comparison))
ax.barh(y - 0.2, comparison['accuracy'], height=0.4, label='accuracy', color='#1f4e79')
ax.barh(y + 0.2, comparison['F1 negative'], height=0.4, label='F1 negative', color='#c0504d')
ax.set_yticks(y, comparison.index)
ax.axvline((y_true == 'positive').mean(), color='gray', ls='--', lw=1, label='always "positive"')
ax.set_xlim(0.4, 1.0); ax.legend(loc='upper left', bbox_to_anchor=(1.01, 1)); ax.set_title('1,352 test reviews (4-5 vs 1-2 stars)')
plt.tight_layout()
plt.show()
comparison

**🤔 Quick check** The three **counting** methods are at the bottom and the three methods with **learned weights** at the top.
The corpus dictionary and the Elastic Net lexicon learn from the same ratings, but they select words differently (a t-test
per word vs a penalized regression) and combine them differently (one vote per word vs a learned weight): 0.798 vs 0.939.
In this sample every method does worse on the **negative** class, and that is where the weights help most.

## &nbsp;Summary

| Lecture | Code | What we saw |
|---|---|---|
| 10-2 p. 24-25 | `MultinomialNB`, `LogisticRegression` | sentiment is a text classification problem: accuracy about 0.96 |
| 10-2 p. 25 | `ngram_range=(1, 2)` | bigrams keep the negation with its word: *not recommend*, *no problem* |
| 11 p. 22-24 | `ElasticNetCV`, `en.coef_` | regression coefficients are sentiment scores, and most of them are exactly 0 |
| 11 p. 29-35 | `spread`, `kneighbors_graph` | 30 seed words spread to a lexicon of 6,485 words |
| 11 p. 41, 44-45 | `lexicon_classify`, `confusion` | evaluation by word and by review |